<a href="https://colab.research.google.com/github/devanshh019/codeexpo/blob/main/competition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [41]:
df = pd.read_csv("/content/emergency_call_dataset_10000_realistic.csv");

In [5]:
df.head()

,call_id,call_text,emergency_type,emotion,sound_event,urgency_level
0,1,oh my god a man is yelling and hitting someone...,domestic_violence,fear,scream,high
1,2,listen flood water is entering houses and peop...,natural_disaster,panic,thunder,critical
2,3,can you help a bike hit a pedestrian and the r...,traffic_accident,stress,car_horn,high
3,4,can you help a man just snatched a bag and ran...,robbery,calm,scream,high
4,5,can you help a child fell into the swimming po...,drowning,fear,water_splash,critical


In [6]:
df.isnull().sum()

,0
call_id,0
call_text,0
emergency_type,0
emotion,0
sound_event,0
urgency_level,0


In [7]:
df['call_text'].duplicated().sum()

df.drop_duplicates(subset=['call_text'], inplace=True)

In [8]:
df.shape

(780, 6)

In [9]:
df['emergency_type'].unique()

array(['domestic_violence', 'natural_disaster', 'traffic_accident',
       'robbery', 'drowning', 'medical_emergency', 'suspicious_activity',
       'shooting', 'fire', 'non_emergency', 'construction_accident',
       'explosion', 'animal_attack'], dtype=object)

In [42]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['emergency_type'] = le.fit_transform(df['emergency_type'])

In [43]:
df.head()

,call_id,call_text,emergency_type,emotion,sound_event,urgency_level
0,1,oh my god a man is yelling and hitting someone...,2,fear,scream,high
1,2,listen flood water is entering houses and peop...,7,panic,thunder,critical
2,3,can you help a bike hit a pedestrian and the r...,12,stress,car_horn,high
3,4,can you help a man just snatched a bag and ran...,9,calm,scream,high
4,5,can you help a child fell into the swimming po...,3,fear,water_splash,critical


# convert in lower case

In [44]:
df['call_text'] = df['call_text'].apply( lambda x: x.lower())

# remove puncuation

In [45]:
import string
def remove_punc(txt):
    return txt.translate(str.maketrans('','',string.punctuation))
df['call_text'] = df['call_text'].apply(remove_punc)

# remove numbers

In [46]:
def remove_numbers(txt):
    new =''
    for i in txt :
        if not i.isdigit():
            new = new + i
    return new

df['call_text'] = df['call_text'].apply(remove_numbers)

# remove emojis

In [47]:
def remove_emojis(txt):
    new = ''
    for i in txt:
        if i.isascii():
            new += i
    return new
df['call_text'] = df['call_text'].apply(remove_emojis)

# remove stopwards

In [48]:
import nltk

In [49]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [50]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [51]:
stop_words = set(stopwords.words('english'))

In [52]:
df.loc[1]['call_text']

'listen flood water is entering houses and people are trapped please send help quickly'

In [53]:
def remove(txt):
  words = txt.split()
  cleaned = []
  for i in words:
    if not i in stop_words:
      cleaned.append(i)

  return ' '.join(cleaned)



In [54]:
df['call_text'] = df['call_text'].apply(remove)

In [55]:
df.loc[1]['call_text']

'listen flood water entering houses people trapped please send help quickly'

In [56]:
y=df['emergency_type']
x=df['call_text']

In [57]:
from sklearn.model_selection import train_test_split


In [58]:
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

In [59]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

vectorizer = TfidfVectorizer(
    min_df=1,
    max_features=5000,
    ngram_range=(1,2)
)



x_train = vectorizer.fit_transform(x_train).toarray()
x_test = vectorizer.transform(x_test).toarray()


In [64]:
'mistake' in vectorizer.vocabulary_

True

In [65]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

model = Sequential([

    Dense(512, activation='relu', input_shape=(x_train.shape[1],)),
    Dropout(0.4),

    Dense(256, activation='relu'),
    Dropout(0.3),

    Dense(128, activation='relu'),  # feature layer for fusion

    Dense(13, activation='softmax')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [67]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [68]:
history = model.fit(
    x_train,
    y_train,
    validation_data=(x_test, y_test),
    epochs=5,
    batch_size=32
)

Epoch 1/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.7423 - loss: 1.1115 - val_accuracy: 1.0000 - val_loss: 9.6382e-05
Epoch 2/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 1.0000 - loss: 5.2751e-04 - val_accuracy: 1.0000 - val_loss: 2.3958e-05
Epoch 3/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 1.0000 - loss: 2.3391e-04 - val_accuracy: 1.0000 - val_loss: 8.8950e-06
Epoch 4/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - accuracy: 1.0000 - loss: 1.1360e-04 - val_accuracy: 1.0000 - val_loss: 4.3239e-06
Epoch 5/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 1.0000 - loss: 6.8451e-05 - val_accuracy: 1.0000 - val_loss: 2.3854e-06


In [79]:
text=input()

text=vectorizer.transform([text])

just a prank call for testing


In [80]:
le.inverse_transform([np.argmax(model.predict(text))])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step


array(['non_emergency'], dtype=object)

In [82]:
model.save('text_model.keras')

In [81]:
import pickle

pickle.dump(vectorizer, open("vectorizer_text.pkl","wb"))

In [ ]:
pickle.dump(le, open("label_encoder_text.pkl","wb"))